# 12. Error Handling, Custom Exceptions & Debugging: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **12. Error Handling, Custom Exceptions & Debugging**. Python uses an exception hierarchy rooted at `BaseException` (and standard errors at `Exception`). This notebook covers structured `try / except / else / finally` blocks, exception chaining via `raise ... from ...`, custom domain exception hierarchies, Exception Groups and `except*` (Python 3.11+), and traceback inspection.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Exception Block: `try`
- [x] 🔹 Exception Interception: `except (Err1, Err2) as e`
- [x] 🔹 Exception Success Clause: `else`
- [x] 🔹 Guaranteed Cleanup: `finally`
- [x] 🔹 Raising Exceptions: `raise`
- [x] 🔹 Exception Chaining: `raise ... from ...`
- [x] 🔹 Custom Exception Hierarchies
- [x] 🔹 Defensive Debugging: `assert`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Exception Block: `try`
- **What it does:** Encloses code blocks that may potentially raise runtime exceptions.
- **Syntax:** `try`
  - **Parameters:**
    - `row_indexer` (*scalar, slice, list, or boolean mask*): Row identifier(s).
  - **Optional Parameters:**
    - `col_indexer` (*scalar, slice, list, or boolean mask*): Column identifier(s).
- **Key Note:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.
- **Dataset Application & Code Demonstration:** Applies Exception Block on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [2]:
try:
    amt = float(transactions[0]['transaction_amount'])
    print('Successfully parsed amount in try block:', amt)
except Exception as e:
    print('Error:', e)

Successfully parsed amount in try block: 607.78


### 🔹 Exception Interception: `except (Err1, Err2) as e`
- **What it does:** Catches and handles specific exception classes, capturing exception instances as `e`.
- **Syntax:** `except (Err1, Err2) as e`
- **Key Note:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.
- **Dataset Application & Code Demonstration:** Demonstrates Exception Interception with practical fintech data structures and variables in the following code block.


In [3]:
try:
    bad_val = float('NOT_A_NUMBER')
except (ValueError, TypeError) as err:
    print('Intercepted specific parsing error:', err)

Intercepted specific parsing error: could not convert string to float: 'NOT_A_NUMBER'


### 🔹 Exception Success Clause: `else`
- **What it does:** `else:` executes ONLY if the code inside the `try` block ran without raising any exceptions.
- **Syntax:** `else`
- **Key Note:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.
- **Dataset Application & Code Demonstration:** Applies Exception Success Clause on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [4]:
try:
    amt = float(transactions[0]['transaction_amount'])
except ValueError:
    print('Error')
else:
    print(f'Else clause executed: Successfully verified amount ${amt:.2f}')

Else clause executed: Successfully verified amount $607.78


### 🔹 Guaranteed Cleanup: `finally`
- **What it does:** `finally:` ALWAYS executes regardless of whether exceptions were raised, caught, or unhandled.
- **Syntax:** `finally`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Demonstrates Guaranteed Cleanup with practical fintech data structures and variables in the following code block.


In [5]:
try:
    x = 10 / 2
finally:
    print('Finally cleanup block executed unconditionally.')

Finally cleanup block executed unconditionally.


### 🔹 Raising Exceptions: `raise`
- **What it does:** Explicitly throws an exception instance halting execution.
- **Syntax:** `raise`
- **Key Note:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.
- **Dataset Application & Code Demonstration:** Applies Raising Exceptions on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [6]:
try:
    if float(transactions[0]['transaction_amount']) < 0:
        raise ValueError('Negative amount illegal!')
    else:
        print('Amount verified non-negative.')
except ValueError as e:
    print('Caught:', e)

Amount verified non-negative.


### 🔹 Exception Chaining: `raise ... from ...`
- **What it does:** Chains a high-level domain error to an underlying root cause exception (PEP 3134).
- **Syntax:** `raise ... from ...`
- **Key Note:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.
- **Dataset Application & Code Demonstration:** Demonstrates Exception Chaining with practical fintech data structures and variables in the following code block.


In [7]:
class BankingServiceError(Exception): pass

try:
    try:
        int('INVALID_ID')
    except ValueError as root_cause:
        raise BankingServiceError('Failed to parse customer identifier') from root_cause
except BankingServiceError as caught:
    print(f'Chained Error: {caught}')
    print(f'Root Cause: {caught.__cause__}')

Chained Error: Failed to parse customer identifier
Root Cause: invalid literal for int() with base 10: 'INVALID_ID'


### 🔹 Custom Exception Hierarchies
- **What it does:** Defines application-specific exception domain models inheriting from base `Exception`.
- **Syntax:** `function(*args, **kwargs)`
- **Key Note:** Always catch specific exceptions (like `ValueError` or `FileNotFoundError`) rather than a bare `except:` to avoid masking unintended bugs.
- **Dataset Application & Code Demonstration:** Demonstrates Custom Exception Hierarchies with practical fintech data structures and variables in the following code block.


In [8]:
class FintechError(Exception):
    """Base exception for all financial system errors."""
    pass

class LimitExceededError(FintechError):
    def __init__(self, tx_id, limit):
        super().__init__(f'Tx {tx_id} exceeds limit ${limit}')
        self.tx_id = tx_id

err = LimitExceededError('TX100', 5000)
print('Custom exception created:', err)

Custom exception created: Tx TX100 exceeds limit $5000


### 🔹 Defensive Debugging: `assert`
- **What it does:** Evaluates internal debugging invariants (`assert cond, message`).
- **Syntax:** `assert`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.
- **Dataset Application & Code Demonstration:** Applies Defensive Debugging on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [9]:
amt = float(transactions[0]['transaction_amount'])
assert amt >= 0.0, f'Amount must be non-negative! Got {amt}'
print(f'Assertion passed: ${amt:.2f} >= 0')

Assertion passed: $607.78 >= 0


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Exception Handling Overhead in Python (EAFP vs LBYL)
- **Objective:** Q1: Exception Handling Overhead in Python (EAFP vs LBYL)
- **Approach:** Explain Python's EAFP ('Easier to Ask for Forgiveness than Permission') philosophy vs LBYL ('Look Before You Leap'). In Python 3.11+ zero-cost exception handling makes try blocks free when no exception is raised.
- **Syntax:** `try: d[key] except KeyError: default` vs `if key in d:`

In [10]:
print('EAFP: try/except is Pythonic and optimized via zero-cost exception tables in Python 3.11+.')

EAFP: try/except is Pythonic and optimized via zero-cost exception tables in Python 3.11+.
